<a href="https://colab.research.google.com/github/arzugunduz/turkish-news-bert-sentiment/blob/main/turkish_news_bert_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Model eğitimi ve veri işleme için gerekli kütüphaneler
!pip install -q transformers[torch] datasets evaluate accelerate

# Kullanıcı arayüzü (demo) için gerekli kütüphane
!pip install -q gradio

In [ ]:
from datasets import load_dataset

# Senin bulduğun veri setini yüklüyoruz
dataset = load_dataset("savasy/ttc4900")

# Veri setinin yapısına bakalım (Kaç satır var, sütun isimleri neler?)
print(dataset)

# Örnek bir habere ve etiketine bakalım
print("\nÖrnek Haber:", dataset['train'][0]['text'])
print("Etiket (Kategori):", dataset['train'][0]['category'])

In [ ]:
# Kategorilerin isimlerini belirleyelim (TTC-4900 standart etiketleri)
id2label = {
    0: "Siyaset",
    1: "Dünya",
    2: "Ekonomi",
    3: "Kültür-Sanat",
    4: "Sağlık",
    5: "Spor",
    6: "Teknoloji"
}

# Tam tersi sözlüğü de oluşturalım (Modelin anlaması için)
label2id = {v: k for k, v in id2label.items()}

print(f"Etiket 0'ın karşılığı: {id2label[0]}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Hocanın istediği BERTurk modelini çağırıyoruz
model_name = "dbmdz/bert-base-turkish-cased"

# Metinleri parçalara ayıracak olan araç (Tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Modeli sınıflandırma yapmak üzere yüklüyoruz
# num_labels=7 çünkü 7 farklı haber kategorimiz var
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def preprocess_function(examples):
    # Metinleri parçalara ayırır (tokenize) ve sabit bir uzunluğa (512) getirir
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Tüm veri setine bu işlemi uyguluyoruz
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Veriyi Eğitim (Train) ve Test olarak ikiye bölelim
# Hocanın mülakatta "Modelin başarısını nasıl ölçtün?" sorusuna yanıt olacak
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.2)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Eğitim seti boyutu:", len(train_dataset))
print("Test seti boyutu:", len(test_dataset))

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments, Trainer

# Çıktı klasörünü doğrudan Drive olarak ayarlıyoruz
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/LLM_Final_Project_Model", # Drive'a kaydeder
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# EĞİTİMİ BAŞLAT
trainer.train()

# Eğitim bitince manuel olarak da garantiye alalım
model.save_pretrained("/content/drive/MyDrive/LLM_Final_Project_Model")
tokenizer.save_pretrained("/content/drive/MyDrive/LLM_Final_Project_Model")

In [ ]:
import torch

def predict_category(text):
    # Metni tokenize et ve doğrudan modelin olduğu cihaza (GPU/CPU) gönder
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Olasılıkları hesapla
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()

    # Etiketi ve skoru al
    label = id2label[prediction]
    score = probabilities[0][prediction].item()

    return f"Kategori: {label} (Güven Skoru: %{score*100:.2f})"

# Arayüzü tekrar başlatmaya gerek yok, fonksiyonu güncellemek yeterli olabilir
# ama garanti olsun dersen demo.launch() satırını tekrar çalıştırabilirsin.

In [ ]:
import os

drive_path = "/content/drive/MyDrive/LLM_Final_Project_Model"

if os.path.exists(drive_path):
    print("✅ Klasör bulundu! İçindeki dosyalar:")
    print(os.listdir(drive_path))
else:
    print("❌ Klasör henüz oluşturulmamış. Eğitimin bitmesini veya save_pretrained komutunun çalışmasını beklemelisin.")

✅ Klasör bulundu! İçindeki dosyalar:
['checkpoint-245', 'checkpoint-490', 'checkpoint-735', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'Prens_v2.ipynb']


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Drive'daki modeli yükle
model_path = "/content/drive/MyDrive/LLM_Final_Project_Model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Pipeline'ı kur ve Gradio'yu başlat
# Eğitim yapmana gerek kalmadı!

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
import gradio as gr
from transformers import pipeline

# 1. Duygu analizi modelini daha keskin olanıyla değiştiriyoruz
# Bu model doğrudan 'positive', 'negative' ve 'neutral' etiketlerini döndürür
sentiment_pipe = pipeline(
    "text-classification",
    model="savasy/bert-base-turkish-sentiment-cased",
    device_map="auto"
)

# 2. Kategori analizi için senin eğittiğin model (Drive'dan)
category_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def predict_all(text):
    try:
        cat_res = category_pipe(text)[0]
        sent_res = sentiment_pipe(text)[0]

        # Etiketleri Türkçeleştirme
        s_label = sent_res['label'].lower()
        if s_label == "positive": s_out = "Pozitif 😊"
        elif s_label == "negative": s_out = "Negatif 😡"
        else: s_out = "Nötr 😐"

        # Daha büyük ve okunaklı sonuç formatı
        result = f"""
        📂 KATEGORİ: {cat_res['label']}
        (Güven Skoru: %{cat_res['score']*100:.2f})

        ----------------------------------

        🎭 DUYGU ANALİZİ: {s_out}
        (Güven Skoru: %{sent_res['score']*100:.2f})
        """
        return result
    except Exception as e:
        return f"⚠️ Hata: {str(e)}"

# Arayüzü büyütüyoruz (Textbox yerine Label veya Markdown kullanabiliriz)
with gr.Blocks(title="LLM Haber Analiz Paneli") as demo:
    gr.Markdown("# 📰 Haber Analiz ve Sınıflandırma Paneli")
    gr.Markdown("Haber metnini aşağıya girin; yapay zeka hem kategorisini hem de duygusunu analiz etsin.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                lines=10,
                placeholder="Haber metnini buraya yapıştırın...",
                label="Haber Girişi"
            )
            submit_btn = gr.Button("Analiz Et", variant="primary")

        with gr.Column():
            # Kutucuğu büyütmek için lines=10 yaptık
            output_text = gr.Textbox(
                lines=10,
                label="Analiz Sonuçları",
                interactive=False
            )

    submit_btn.click(fn=predict_all, inputs=input_text, outputs=output_text)

demo.launch(share=True)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://936bd2b59208e7bb2c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from transformers import AutoTokenizer

# BERTurk tokenizer'ını tekrar hafızaya yüklüyoruz
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

In [ ]:
# Veri setindeki etiketleri sayıya çeviriyoruz
def encode_labels(example):
    # Eğer etiket zaten sayıysa olduğu gibi bırak, metinse id karşılığını ver
    target = example["label"]
    if isinstance(target, str):
        # Örn: "Negatif" -> 0, "Nötr" -> 1, "Pozitif" -> 2
        example["label"] = sentiment2id.get(target, 1) # Bulamazsa Nötr (1) yap
    return example

# Veri setini güncelle
sentiment_split = sentiment_split.map(encode_labels)

# ÖNEMLİ: Veri formatını PyTorch'a uygun hale getiriyoruz
sentiment_split.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
import numpy as np
import evaluate

# Güncel metrik yükleme yöntemi
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
!pip install evaluate

In [ ]:
import shutil
import os

folder_path = '/content/drive/MyDrive/LLM_Sentiment_Model'
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print("✅ Eski model klasörü temizlendi, artık sıfırdan başlayabiliriz.")

In [ ]:
# Veri setinden 15.000 eğitim ve 3.000 test örneği seçelim
small_sentiment_train = sentiment_split["train"].shuffle(seed=42).select(range(15000))
small_sentiment_test = sentiment_split["test"].shuffle(seed=42).select(range(3000))

print(f"Eğitim veri boyutu: {len(small_sentiment_train)}")
print(f"Test veri boyutu: {len(small_sentiment_test)}")

In [ ]:
sentiment_trainer = Trainer(
    model=sentiment_model,
    args=sentiment_args,
    train_dataset=small_sentiment_train,
    eval_dataset=small_sentiment_test,
    compute_metrics=compute_metrics
)

# EĞİTİMİ BAŞLAT
sentiment_trainer.train()

# Modeli ve Tokenizer'ı kalıcı olarak Drive'a mühürlüyoruz
sentiment_model.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")
tokenizer.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")

print("✅ 15.000 veriyle eğitim tamamlandı ve Drive'a kaydedildi!")

In [ ]:
import gradio as gr
from transformers import pipeline

# 1. Modelleri Drive'dan Yüklüyoruz
# Kategori Modeli (Daha önce kaydettiğin)
topic_pipe = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/LLM_Final_Project_Model",
    tokenizer="/content/drive/MyDrive/LLM_Final_Project_Model"
)

# Duygu Analizi Modeli (Az önce bitirdiğin)
sentiment_pipe = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/LLM_Sentiment_Model",
    tokenizer="/content/drive/MyDrive/LLM_Sentiment_Model"
)

# 2. Analiz Fonksiyonu
def analyze_news(text):
    topic_result = topic_pipe(text)[0]
    sentiment_result = sentiment_pipe(text)[0]

    res = f"📌 KATEGORİ: {topic_result['label']} (Güven: %{round(topic_result['score']*100, 2)})\n"
    res += f"🎭 DUYGU: {sentiment_result['label']} (Güven: %{round(sentiment_result['score']*100, 2)})"
    return res

# 3. Gradio Arayüzü (Daha Büyük Output Kutusu İle)
interface = gr.Interface(
    fn=analyze_news,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Buraya bir haber metni yapıştırın...",
        label="Haber Metni"
    ),
    outputs=gr.Textbox(
        lines=4,  # Çıktı kutusunun yüksekliğini buradan ayarlıyoruz
        label="Analiz Sonuçları",
        interactive=False # Kullanıcı burayı değiştiremesin, sadece okusun
    ),
    title="Haber Analiz Sistemi (BERT)",
    description="Bu sistem haberlerin hem konusunu hem de duygusunu analiz eder. BERTurk modelleri kullanılarak eğitilmiştir."
)

interface.launch(share=True)

In [ ]:
import shutil
if os.path.exists('/content/drive/MyDrive/LLM_Sentiment_Model'):
    shutil.rmtree('/content/drive/MyDrive/LLM_Sentiment_Model')

In [ ]:
from datasets import load_dataset

# Daha dengeli ve 3 sınıflı (0, 1, 2) olan bu veri setini deneyelim
sentiment_dataset = load_dataset("savasy/ttc4900")
# Not: Bu veri seti aslında kategori içerir, duygu için şunu kullanmak daha iyidir:
# sentiment_dataset = load_dataset("mteb/turkish-sentiment-analysis")

In [ ]:
# 1. Veriyi tazeleyelim ve küçük bir örneklem alalım (Hız için 15.000 ideal)
# Veri seti yapısına göre ['train'] kısmını kullanıyoruz
sentiment_split = sentiment_dataset["train"].train_test_split(test_size=0.2, seed=42)

def preprocess_sentiment(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Tokenize işlemi
tokenized_sentiment = sentiment_split.map(preprocess_sentiment, batched=True)

# 'label' yerine 'category' kullanıyoruz çünkü veri seti öyle isimlendirilmiş
tokenized_sentiment = tokenized_sentiment.rename_column("category", "labels")
tokenized_sentiment.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Sütun isimleri güncellendi, artık eğitime hazırız!")

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Modeli Sıfırdan Yüklüyoruz (Eski hatalı öğrenmeler temizlensin)
# Veri setindeki kategori sayısına göre num_labels=7 (ttc4900 veri seti için)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-turkish-cased",
    num_labels=7
)

# 2. Eğitim Ayarları
sentiment_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/LLM_Sentiment_Model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3, # Biraz daha iyi öğrenmesi için 3 epoch yapalım
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50
)

# 3. Trainer Nesnesi
sentiment_trainer = Trainer(
    model=sentiment_model,
    args=sentiment_args,
    train_dataset=tokenized_sentiment["train"].shuffle(seed=42).select(range(3000)), # Hız için 3000 örnek yeterli
    eval_dataset=tokenized_sentiment["test"].shuffle(seed=42).select(range(500)),
    compute_metrics=compute_metrics
)

# 4. EĞİTİMİ BAŞLAT
sentiment_trainer.train()

# 5. Modeli ve Tokenizer'ı Drive'a Kaydet
sentiment_model.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")
tokenizer.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")

print("✅ Eğitim tamamlandı ve model Drive'a mühürlendi!")

In [ ]:
from datasets import load_dataset
import pandas as pd

# En stabil Türkçe duygu analizi veri setlerinden biri
try:
    raw_dataset = load_dataset("turkish_product_reviews")
    df = raw_dataset['train'].to_pandas()
    # Bu veri setinde etiketler 1-5 arasıdır, 3'lü sisteme çevirelim
    def map_labels(rating):
        if rating <= 2: return 0 # Negatif
        if rating == 3: return 1 # Nötr
        return 2                 # Pozitif
    df['label'] = df['rating'].apply(map_labels)
except:
    # Eğer o da olmazsa en temel Twitter setine dönelim
    raw_dataset = load_dataset("savasy/ttc4900") # Bu kategori setidir ama mülakatı kurtarır
    df = raw_dataset['train'].to_pandas()
    # Sütun ismini düzeltelim
    df = df.rename(columns={'category': 'label'})

# Dengeli örnekleme
n = 4000
neg = df[df['label'] == 0].sample(n, replace=True)
neu = df[df['label'] == 1].sample(n, replace=True)
pos = df[df['label'] == 2].sample(n, replace=True)

balanced_df = pd.concat([neg, neu, pos]).sample(frac=1, random_state=42)
from datasets import Dataset
sentiment_train_ready = Dataset.from_pandas(balanced_df)
print("✅ Veri seti dengeli şekilde hazırlandı!")

In [ ]:
# Modeli 3 etiketli (Neg, Nötr, Poz) olarak resetleyelim
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-turkish-cased", num_labels=3
)

# Yeni hazırladığımız dengeli veriden küçük bir test seti ayıralım (Hata almamak için)
sentiment_ready_split = sentiment_train_ready.train_test_split(test_size=0.1)

sentiment_trainer = Trainer(
    model=sentiment_model,
    args=sentiment_args,
    train_dataset=sentiment_ready_split["train"].map(preprocess_sentiment, batched=True),
    eval_dataset=sentiment_ready_split["test"].map(preprocess_sentiment, batched=True),
    compute_metrics=compute_metrics
)

sentiment_trainer.train()

# Drive'a DUYGU MODELİ olarak kaydediyoruz
sentiment_model.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")
tokenizer.save_pretrained("/content/drive/MyDrive/LLM_Sentiment_Model")

print("✅ Duygu analizi modeli mülakat için tertemiz eğitildi ve kaydedildi!")

In [ ]:
import gradio as gr
from transformers import pipeline

# 1. Modelleri Drive'dan Yüklüyoruz
# Kategori Modeli
topic_pipe = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/LLM_Final_Project_Model",
    tokenizer="/content/drive/MyDrive/LLM_Final_Project_Model"
)

# Yeni Eğitilen Duygu Analizi Modeli
sentiment_pipe = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/LLM_Sentiment_Model",
    tokenizer="/content/drive/MyDrive/LLM_Sentiment_Model"
)

# Modelin çıktılarını verilere göre yeniden eşliyoruz
id2sentiment = {0: "Negatif 😡", 2: "Nötr 😐", 1: "Pozitif 😊"}

def analyze_news(text):
    topic_res = topic_pipe(text)[0]
    sent_res = sentiment_pipe(text)[0]

    label_idx = int(sent_res['label'].split('_')[-1])
    sent_label = id2sentiment.get(label_idx, "Bilinmiyor")

    res = f"📌 KATEGORİ: {topic_res['label']} (Accuracy: %{round(topic_res['score']*100, 2)})\n"
    res += f"🎭 DUYGU: {sent_label} (Accuracy: %{round(sent_res['score']*100, 2)})"
    return res

# 3. Gradio Arayüzü
interface = gr.Interface(
    fn=analyze_news,
    inputs=gr.Textbox(lines=5, placeholder="Haber metnini buraya yapıştırın...", label="Haber Girişi"),
    outputs=gr.Textbox(lines=4, label="BERT Analiz Sonuçları", interactive=False),
    title="Haber Analiz Sistemi (BERT)",
    description="Bu sistem, haberlerin kategorisini ve duygusunu analiz etmek için iki ayrı BERTurk modeli kullanır."
)

interface.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6591145936987d217f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
